# 04 - Baseline vs cliff-aware (the headline)

Aggregate metrics hide the failure that matters: most predictors collapse on the
single-residue flips. Here we train a **baseline** (embeddings + gradient boosting,
no torch) and, **if torch is available**, a **cliff-aware** contrastive model, then
compare them with `eval.compare_models` - paying attention to the **cliff-subset
performance gap**.

In [ ]:
import numpy as np
from tcr_cliff.data import load_toy, split_pairs
from tcr_cliff.config import Config, DataConfig, EmbeddingConfig, CliffConfig, ModelConfig
from tcr_cliff.models import train_model, predict_scores
from tcr_cliff.eval import cliff_aware_report, compare_models
from tcr_cliff.cliffs import find_neighbor_pairs

df = load_toy()
parts = split_pairs(df, DataConfig(group_split_on='peptide', split_column='__none__'), seed=0)
train_df, test_df = parts['train'], parts['test']
print('train/test rows:', len(train_df), len(test_df))

## Shared config (small + offline)

Fallback embedder with a small dimension keeps everything fast and dependency-free.

In [ ]:
base_cfg = Config(
    seed=0,
    embedding=EmbeddingConfig(backend='fallback', fallback_dim=64, cache_dir=None),
    cliff=CliffConfig(max_edits=1, vary='both', require_label_flip=True),
    model=ModelConfig(kind='baseline_lgbm'),
)
# Smaller baseline for the demo (transparently uses sklearn HistGBM if LightGBM absent).
base_cfg.model.baseline.n_estimators = 50

# Detect neighbour pairs on the TEST split once; reuse for every model's report.
test_pairs = find_neighbor_pairs(test_df, base_cfg.cliff)
print('test neighbour pairs:', len(test_pairs),
      '| cliffs:', sum(p.is_cliff for p in test_pairs))

## Train the baseline and score the test set

In [ ]:
baseline = train_model(base_cfg, train_df)
y_base = predict_scores(baseline, test_df)
rep_base = cliff_aware_report(test_df, y_base, pairs=test_pairs,
                              threshold=base_cfg.eval.threshold)
print('baseline overall AUROC     :', round(rep_base['overall']['auroc'], 3))
print('baseline cliff-record AUROC:', round(rep_base['cliff_records']['auroc'], 3))

## Train the cliff-aware model **if torch is available**

The cliff-aware encoder is contrastively pretrained to *pull smooth neighbours
together and push cliff pairs apart*, then a supervised head is fit on top. If
torch is missing (offline minimal install) we skip gracefully and explain.

In [ ]:
reports = {'baseline': rep_base}

try:
    import torch  # noqa: F401
    torch_available = True
except ImportError:
    torch_available = False

if torch_available:
    ca_cfg = base_cfg.model_copy(deep=True)
    ca_cfg.model = ModelConfig(kind='cliff_aware')
    # Keep the demo tiny.
    ca_cfg.model.cliff_aware.proj_dim = 32
    ca_cfg.model.cliff_aware.hidden_dim = 64
    ca_cfg.model.cliff_aware.contrastive_epochs = 3
    ca_cfg.model.cliff_aware.head_epochs = 5
    cliff_aware = train_model(ca_cfg, train_df)
    y_ca = predict_scores(cliff_aware, test_df)
    reports['cliff_aware'] = cliff_aware_report(
        test_df, y_ca, pairs=test_pairs, threshold=ca_cfg.eval.threshold
    )
    print('cliff-aware trained; cliff-record AUROC:',
          round(reports['cliff_aware']['cliff_records']['auroc'], 3))
else:
    print('torch not available - skipping cliff-aware model.')
    print('Install with:  pip install "tcr-cliff[torch]"')
    print('The cliff-aware model contrastively separates cliff vs smooth neighbour')
    print('pairs, which specifically lifts performance on the cliff subset below.')

## The headline: cliff-subset performance gap

`compare_models` lays each model's **overall AUROC** next to its **cliff-record
AUROC**, the **gap** between non-cliff and cliff records, and the **cliff-pair
directional accuracy** (does the model rank the binder above the non-binder within
each flip pair? random = 0.5).

In [ ]:
table = compare_models(reports)
table

In [ ]:
# The story in one number: how much AUROC is lost specifically on cliff records.
for name, rep in reports.items():
    overall = rep['overall']['auroc']
    cliff = rep['cliff_records']['auroc']
    gap = rep['gap']['auroc_gap']
    diracc = rep['pair_level']['cliff_pair_directional_accuracy']
    print(f'{name:12s} overall={overall:.3f} cliff={cliff:.3f} '
          f'gap(non-cliff - cliff)={gap:.3f} dir-acc={diracc:.3f}')

**Reading the table.** A large positive `auroc_gap` means the model is much weaker
on cliff records than on smooth ones - the failure mode this package exposes. The
cliff-aware model is designed to *shrink that gap* and push directional accuracy
above 0.5 on cliff pairs.

### Next
Continue to **05_structure_fusion** to add AlphaFold-confidence features.